# Notebook 06 — Model Evaluation

**Chapter 3, Section 3.5.5 — Model Evaluation**

## Objectives
- Evaluate the trained LSTM on the held-out test set
- Compute all metrics: Accuracy, Precision, Recall, F1, ROC-AUC
- Generate the Classification Report
- Plot Confusion Matrix and ROC Curves
- Compute Permutation Feature Importance

---

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import json

from src.utils.helpers import set_global_seed, print_banner
from src.config import get_config
from src.utils.paths import PROCESSED_DATA_DIR, FINAL_MODEL_DIR
from src.utils.serialization import load_processed_arrays, load_keras_model
from src.utils.constants import FINAL_MODEL_KERAS, METADATA_JSON

cfg = get_config()
set_global_seed(cfg.seed)
print_banner('Notebook 06 — Model Evaluation')

## 1. Load Test Data and Model

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = \
    load_processed_arrays(PROCESSED_DATA_DIR)

with open(FINAL_MODEL_DIR / METADATA_JSON) as f:
    metadata = json.load(f)

class_names = metadata['class_names']
n_classes   = metadata['n_classes']

model = load_keras_model(FINAL_MODEL_DIR / FINAL_MODEL_KERAS)
print(f'X_test : {X_test.shape}')
print(f'Model  : {model.name}  ({model.count_params():,} params)')

## 2. Generate Predictions

In [ ]:
from src.evaluation.metrics import predict_lstm

y_pred, y_prob = predict_lstm(model, X_test)
print(f'y_pred shape : {y_pred.shape}')
print(f'y_prob shape : {y_prob.shape}')
print(f'Unique predicted classes: {np.unique(y_pred).tolist()}')

## 3. Compute All Metrics

In [ ]:
from src.evaluation.metrics import compute_metrics

metrics = compute_metrics(
    y_test, y_pred, y_prob,
    class_names=class_names,
    dataset='nsl_kdd',
    model_name='LSTM',
)

print(f'Accuracy          : {metrics["accuracy"]:.4f}')
print(f'Precision (macro) : {metrics["precision_macro"]:.4f}')
print(f'Recall    (macro) : {metrics["recall_macro"]:.4f}')
print(f'F1-Score  (macro) : {metrics["f1_macro"]:.4f}')
print(f'F1-Score  (wtd)   : {metrics["f1_weighted"]:.4f}')
if metrics['roc_auc']:
    print(f'ROC-AUC           : {metrics["roc_auc"]:.4f}')

## 4. Per-Class Metrics

In [ ]:
print(f'Per-class metrics:')
print(f'  {"Class":<12}  {"Precision":>10}  {"Recall":>10}  {"F1":>10}  {"Support":>10}')
print('  ' + '-' * 58)
for cls_name, m in metrics['per_class_metrics'].items():
    print(f'  {cls_name:<12}  {m["precision"]:>10.4f}  {m["recall"]:>10.4f}  '
          f'{m["f1_score"]:>10.4f}  {m["support"]:>10,}')

## 5. Classification Report

In [ ]:
from src.evaluation.classification_report import generate_classification_report

report = generate_classification_report(
    y_test, y_pred,
    class_names=class_names,
    dataset='nsl_kdd',
    model_name='LSTM',
)
print(report['report_text'])

## 6. Confusion Matrix

In [ ]:
from src.evaluation.confusion_matrix import plot_confusion_matrix
from IPython.display import Image

path = plot_confusion_matrix(
    y_test, y_pred,
    class_names=class_names,
    dataset='nsl_kdd',
    model_name='LSTM',
    normalize=True,
)
print(f'Confusion matrix saved: {path}')
Image(str(path))

## 7. ROC Curves

In [ ]:
from src.evaluation.roc_analysis import (
    compute_roc_curves, plot_roc_curves, save_roc_scores
)

roc_data = compute_roc_curves(y_test, y_prob, class_names, 'nsl_kdd')

print('AUC scores per class:')
for cls_name, data in roc_data['per_class'].items():
    print(f'  {cls_name:<12}: AUC = {data["auc"]:.4f}')
if 'auc' in roc_data.get('macro', {}):
    print(f'  {"Macro avg":<12}: AUC = {roc_data["macro"]["auc"]:.4f}')

path = plot_roc_curves(y_test, y_prob, class_names=class_names, dataset='nsl_kdd')
save_roc_scores(roc_data)
Image(str(path))

## 8. Permutation Feature Importance

In [ ]:
from src.data.feature_engineering import (
    compute_permutation_importance, plot_feature_importance
)
from src.utils.serialization import load_feature_names

feature_names = load_feature_names(FINAL_MODEL_DIR / 'feature_names.pkl')

importance_df = compute_permutation_importance(
    model, X_test[:2000], y_test[:2000],  # Use subset for speed
    feature_names=feature_names,
    n_repeats=5,
)

print('Top 10 most important features:')
print(importance_df.head(10).to_string(index=False))

path = plot_feature_importance(importance_df, dataset='nsl_kdd', top_n=15)
Image(str(path))

---
**Summary:** LSTM fully evaluated on held-out test set.  
All metrics, confusion matrix, ROC curves, and feature importance saved to `reports/`.  
**Next:** Run `07_results_analysis.ipynb`